# Pregunta 2 — Popularitat i nota prediuen la recaptació?
**Projecte:** Anàlisi d'un conjunt de dades (0378 – Seguretat i alta disponibilitat, Tema 1)
**Dataset:** `IMDB-Movie-Data.csv` — 1.000 pel·lícules (2006–2016)

Aquest notebook treballa **una única pregunta de recerca**, amb el procediment complet: preparació de les dades, mètode d'anàlisi, resultat i interpretació.

## 1. Preparació de les dades

**Procediment:**
- Carreguem el CSV en un `DataFrame` de Pandas.
- Renombrem columnes amb espais/parèntesis (`runtime`, `revenue`).
- Convertim `Revenue = 0` a `NaN` (no és una recaptació real, és un valor desconegut).
- Separem `genre` en una llista de gèneres (una pel·lícula en pot tenir fins a 3).
- Creem un subconjunt `df_fiable` amb `votes >= 1000`, perquè les pel·lícules amb pocs vots tenen una nota poc representativa (sobretot les de 2016).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("IMDB-Movie-Data.csv", encoding="utf-8")
df = df.rename(columns={"Runtime (Minutes)": "runtime", "Revenue (Millions)": "revenue"})
df.columns = [c.lower() for c in df.columns]

# Revenue = 0 és, amb tota probabilitat, un valor desconegut, no una recaptació real
df.loc[df["revenue"] == 0, "revenue"] = np.nan

# Gèneres com a llista (eren text separat per comes)
df["genres"] = df["genre"].str.split(",").apply(lambda l: [g.strip() for g in l])

# Subconjunt amb vots suficients per considerar la nota fiable
df_fiable = df[df["votes"] >= 1000].copy()

print(f"Total de pel·lícules: {len(df)}")
print(f"Amb >= 1000 vots (notes fiables): {len(df_fiable)}")
df.head(3)

## 2. Pregunta de recerca

**Hi ha relació entre el nombre de vots (`Votes`, popularitat a IMDb) i la recaptació (`Revenue`)? I entre la nota (`Rating`) i la recaptació?**

**Mètode:** calculem les correlacions de Pearson i Spearman per a cada parella de variables. Fem servir Spearman com a referència principal perquè `revenue` i `votes` tenen una distribució molt asimètrica (unes poques pel·lícules recapten moltíssim), cosa que distorsiona Pearson. També comparem la recaptació mitjana per trams de nota.

In [ ]:
r = df.dropna(subset=["revenue"]).copy()
print(f"Pel·lícules amb recaptació disponible: {len(r)} de {len(df)}")

print("Votes vs Revenue  -> Pearson: {:.3f} | Spearman: {:.3f}".format(
    r["votes"].corr(r["revenue"]), r["votes"].corr(r["revenue"], method="spearman")))
print("Rating vs Revenue -> Pearson: {:.3f} | Spearman: {:.3f}".format(
    r["rating"].corr(r["revenue"]), r["rating"].corr(r["revenue"], method="spearman")))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
sns.scatterplot(data=r, x="votes", y="revenue", alpha=0.45, s=20, ax=ax[0])
ax[0].set_xscale("log"); ax[0].set_title("Vots (escala log) vs recaptació (M)")
sns.scatterplot(data=r, x="rating", y="revenue", alpha=0.45, s=20, ax=ax[1])
ax[1].set_title("Rating vs recaptació (M)")
plt.tight_layout(); plt.show()

### Recaptació mitjana per trams de nota

In [ ]:
r["tram_rating"] = pd.cut(r["rating"], [0, 5.99, 6.99, 7.99, 10],
                           labels=["< 6", "6 – 6,9", "7 – 7,9", "≥ 8"])
resum = r.groupby("tram_rating", observed=True)["revenue"].agg(
    n="count", mediana="median", mitjana="mean").round(1)
resum

## 3. Resultats

- `Votes` i `Revenue`: Pearson **0,64**, Spearman **0,70** (n = 871). És la correlació més forta de les calculades en aquest projecte: les pel·lícules més vistes/comentades a IMDb solen ser les que més recapten, cosa coherent (ambdues reflecteixen popularitat).
- `Rating` i `Revenue`: Pearson **0,22**, Spearman **0,16**. La relació és feble: **una nota alta no garanteix una bona recaptació**.
- Per trams de nota, la mediana de recaptació puja de manera clara només al tram de notes **≥ 8** (103,7 M enfront de 34–50 M als altres trams): cal una nota molt alta perquè es noti en diners; entre 6 i 8 la diferència és petita.


## 4. Interpretació i limitacions

✏️ **Omple-ho tu:** per exemple, per què `votes` prediu millor la recaptació que `rating`? Pensa en el paper del pressupost de màrqueting o el nombre de sales, variables que aquest dataset no té. Esmenta també que hi ha 129 pel·lícules sense `revenue`, concentrades sobretot a 2016.